In [1]:
!pip install onnxscript -q
!pip install onnxruntime -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 92.9 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torch.optim.lr_scheduler import CosineAnnealingLR

from PIL import Image
from tqdm import tqdm

# Configur

In [3]:
class Config:

    DATA_ROOT       = "/kaggle/input/datasets/msambare/fer2013"
    TRAIN_DIR       = os.path.join(DATA_ROOT, "train")
    VAL_DIR         = os.path.join(DATA_ROOT, "test")

    OUTPUT_DIR      = "/kaggle/working"
    MODEL_SAVE_PATH = "/kaggle/working/vad_multitask.pth"
    ONNX_SAVE_PATH  = "/kaggle/working/vad_multitask.onnx"

    IMAGE_SIZE      = 224
    BATCH_SIZE      = 64
    NUM_EPOCHS      = 40
    LR              = 3e-4
    WEIGHT_DECAY    = 1e-4
    DROPOUT         = 0.4
    SEED            = 42
    FREEZE_EPOCHS   = 10

    # Loss weighting: classification loss weight vs VAD regression loss weight.
    # Classification provides the strong gradient signal, VAD refines on top.
    CLS_WEIGHT      = 0.6
    VAD_WEIGHT      = 0.4

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


cfg = Config()

torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)

print(f"Device    : {cfg.DEVICE}")
print(f"Train dir : {cfg.TRAIN_DIR}")
print(f"Val dir   : {cfg.VAL_DIR}")

Device    : cuda
Train dir : /kaggle/input/datasets/msambare/fer2013/train
Val dir   : /kaggle/input/datasets/msambare/fer2013/test


# VAD Mapping

In [4]:
CLASS_NAMES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES  = len(CLASS_NAMES)

# VAD targets in [-1, 1] per class.
# Source: NRC-VAD Lexicon + VAD-Net paper (arxiv 2512.06377), scale [-2,2] / 2.
# These are class-level soft targets. The classification head provides
# hard label supervision; VAD head learns the continuous mapping.
VAD_MAP = {
    "angry":    (-1.00,  0.75,  0.50),
    "disgust":  (-0.50,  0.25,  0.25),
    "fear":     (-0.75,  0.50, -0.75),
    "happy":    ( 0.75,  0.50,  0.25),
    "neutral":  ( 0.00,  0.00,  0.00),
    "sad":      (-0.75, -0.25, -0.50),
    "surprise": ( 0.25,  0.75, -0.25),
}

print("Class index mapping:")
for name, idx in CLASS_TO_IDX.items():
    vad = VAD_MAP[name]
    print(f"  {idx} | {name:<10} | V={vad[0]:+.2f}  A={vad[1]:+.2f}  D={vad[2]:+.2f}")

Class index mapping:
  0 | angry      | V=-1.00  A=+0.75  D=+0.50
  1 | disgust    | V=-0.50  A=+0.25  D=+0.25
  2 | fear       | V=-0.75  A=+0.50  D=-0.75
  3 | happy      | V=+0.75  A=+0.50  D=+0.25
  4 | neutral    | V=+0.00  A=+0.00  D=+0.00
  5 | sad        | V=-0.75  A=-0.25  D=-0.50
  6 | surprise   | V=+0.25  A=+0.75  D=-0.25


# Dataset

In [5]:
class FERMultiTask(Dataset):
    """
    Returns image, class index (for classification loss),
    and VAD target (for regression loss).

    This multi-task setup follows the approach in the VAD-Net paper:
    a shared backbone feeds both a classification head and a VAD
    regression head simultaneously.
    """

    def __init__(self, root, transform=None, is_train=True):
        self.samples   = []
        self.transform = transform
        self.is_train  = is_train

        for class_name in CLASS_NAMES:
            class_dir = os.path.join(root, class_name)
            if not os.path.isdir(class_dir):
                print(f"Warning: folder not found — {class_dir}")
                continue

            label_idx  = CLASS_TO_IDX[class_name]
            vad_values = VAD_MAP[class_name]
            vad_tensor = torch.tensor(vad_values, dtype=torch.float32)

            for fname in os.listdir(class_dir):
                if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.samples.append((
                        os.path.join(class_dir, fname),
                        label_idx,
                        vad_tensor,
                    ))

        print(f"Loaded {len(self.samples):,} samples from: {root}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label_idx, vad = self.samples[idx]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # Add small gaussian noise to VAD targets during training only.
        # This prevents the model from memorizing 7 fixed values and
        # encourages learning a more continuous output space.
        if self.is_train:
            vad = vad + torch.randn(3) * 0.08
            vad = torch.clamp(vad, -1.0, 1.0)

        return image, label_idx, vad


def build_transforms(image_size):
    train_tf = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        transforms.RandomGrayscale(p=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    val_tf = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    return train_tf, val_tf


def load_data(cfg):
    train_tf, val_tf = build_transforms(cfg.IMAGE_SIZE)

    train_ds = FERMultiTask(cfg.TRAIN_DIR, transform=train_tf, is_train=True)
    val_ds   = FERMultiTask(cfg.VAL_DIR,   transform=val_tf,   is_train=False)

    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE,
                              shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, val_loader

# Model

In [6]:
class MultiTaskVAD(nn.Module):
    """
    EfficientNet-B0 backbone with two output heads:
      1. Classification head  — 7-class softmax (cross-entropy loss)
      2. VAD regression head  — 3 continuous values in [-1, 1] (MSE loss)

    Architecture follows the VAD-Net paper approach: shared backbone,
    parallel heads for discrete classification and continuous VAD regression.
    """

    def __init__(self, num_classes=7, dropout=0.4):
        super().__init__()

        backbone            = models.efficientnet_b0(weights="IMAGENET1K_V1")
        in_features         = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        self.backbone       = backbone

        self.cls_head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, num_classes),
        )

        self.vad_head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout / 2),
            nn.Linear(256, 3),
            nn.Tanh(),
        )

    def forward(self, x):
        features   = self.backbone(x)
        cls_logits = self.cls_head(features)
        vad_output = self.vad_head(features)
        return cls_logits, vad_output

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_last_blocks(self, num_blocks=3):
        blocks = list(self.backbone.features.children())
        for block in blocks[-num_blocks:]:
            for param in block.parameters():
                param.requires_grad = True

# Loss Function

In [7]:
class CombinedLoss(nn.Module):
    """
    Weighted sum of:
      - CrossEntropyLoss for 7-class classification (primary signal)
      - MSELoss for VAD regression (secondary signal)

    Classification loss dominates early training to build good features.
    VAD loss steers the regression head to produce meaningful V, A, D values.
    """

    def __init__(self, cls_weight=0.6, vad_weight=0.4):
        super().__init__()
        self.cls_weight = cls_weight
        self.vad_weight = vad_weight
        self.ce_loss    = nn.CrossEntropyLoss()
        self.mse_loss   = nn.MSELoss()

    def forward(self, cls_logits, vad_pred, cls_targets, vad_targets):
        loss_cls = self.ce_loss(cls_logits, cls_targets)
        loss_vad = self.mse_loss(vad_pred, vad_targets)
        total    = self.cls_weight * loss_cls + self.vad_weight * loss_vad
        return total, loss_cls.item(), loss_vad.item()

# Training & Val Loops

In [8]:
def run_epoch(model, loader, criterion, optimizer, device, is_train=True):
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_cls  = 0.0
    total_vad  = 0.0
    correct    = 0
    total_samples = 0

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for images, cls_targets, vad_targets in tqdm(loader, leave=False):
            images      = images.to(device)
            cls_targets = cls_targets.to(device)
            vad_targets = vad_targets.to(device)

            cls_logits, vad_pred = model(images)
            loss, l_cls, l_vad  = criterion(cls_logits, vad_pred, cls_targets, vad_targets)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss    += loss.item()
            total_cls     += l_cls
            total_vad     += l_vad
            preds          = cls_logits.argmax(dim=1)
            correct       += (preds == cls_targets).sum().item()
            total_samples += len(cls_targets)

    n          = len(loader)
    acc        = correct / total_samples * 100
    return total_loss / n, total_cls / n, total_vad / n, acc


def train(cfg):
    train_loader, val_loader = load_data(cfg)

    model     = MultiTaskVAD(num_classes=NUM_CLASSES, dropout=cfg.DROPOUT).to(cfg.DEVICE)
    criterion = CombinedLoss(cls_weight=cfg.CLS_WEIGHT, vad_weight=cfg.VAD_WEIGHT)
    optimizer = optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.NUM_EPOCHS, eta_min=1e-6)

    best_val_acc = 0.0
    history      = []

    print("\nStarting training...\n")
    header = f"{'Epoch':>6} | {'Phase':>6} | {'Total':>8} | {'CLS':>8} | {'VAD':>8} | {'Acc':>7}"
    print(header)
    print("-" * len(header))

    for epoch in range(1, cfg.NUM_EPOCHS + 1):

        if epoch == 1:
            model.freeze_backbone()
            print("Backbone frozen — training heads only.\n")

        if epoch == cfg.FREEZE_EPOCHS + 1:
            model.unfreeze_last_blocks(num_blocks=3)
            for g in optimizer.param_groups:
                g["lr"] = cfg.LR * 0.5
            print("\nUnfroze last 3 backbone blocks.\n")

        tr_loss, tr_cls, tr_vad, tr_acc = run_epoch(
            model, train_loader, criterion, optimizer, cfg.DEVICE, is_train=True
        )
        vl_loss, vl_cls, vl_vad, vl_acc = run_epoch(
            model, val_loader, criterion, optimizer, cfg.DEVICE, is_train=False
        )

        scheduler.step()

        print(f"{epoch:>6} | {'train':>6} | {tr_loss:>8.4f} | {tr_cls:>8.4f} | {tr_vad:>8.4f} | {tr_acc:>6.2f}%")
        print(f"{'':>6} | {'val':>6} | {vl_loss:>8.4f} | {vl_cls:>8.4f} | {vl_vad:>8.4f} | {vl_acc:>6.2f}%")
        print("-" * len(header))

        history.append({
            "epoch": epoch,
            "train_loss": tr_loss, "val_loss": vl_loss,
            "train_acc": tr_acc,   "val_acc": vl_acc,
            "train_vad_mse": tr_vad, "val_vad_mse": vl_vad,
        })

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), cfg.MODEL_SAVE_PATH)
            print(f"  >> Best model saved (val acc: {best_val_acc:.2f}%)\n")

    print(f"\nTraining complete. Best Val Accuracy: {best_val_acc:.2f}%")
    return model, history

# ONNX export

In [9]:
def export_onnx(model, cfg):
    """
    Exports the trained model to ONNX.
    Input  shape : (1, 3, 224, 224)
    Output shape : (1, 7) cls_logits,  (1, 3) vad values

    At inference time in FaceR-Komplit.py:
      - cls_logits → argmax → emotion label
      - vad        → [valence, arousal, dominance]
    """
    model.eval()
    model.load_state_dict(torch.load(cfg.MODEL_SAVE_PATH, map_location=cfg.DEVICE))
    model.to(cfg.DEVICE)

    dummy = torch.randn(1, 3, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE).to(cfg.DEVICE)

    torch.onnx.export(
        model,
        dummy,
        cfg.ONNX_SAVE_PATH,
        dynamo=False,
        input_names=["face_crop"],
        output_names=["cls_logits", "vad"],
        dynamic_axes={
            "face_crop":  {0: "batch_size"},
            "cls_logits": {0: "batch_size"},
            "vad":        {0: "batch_size"},
        },
        opset_version=11,
        do_constant_folding=True,
    )

    size_mb = os.path.getsize(cfg.ONNX_SAVE_PATH) / (1024 * 1024)
    print(f"ONNX exported : {cfg.ONNX_SAVE_PATH} ({size_mb:.1f} MB)")

# ONNX Sanity Check

In [10]:
def verify_onnx(cfg):
    import onnxruntime as ort

    session = ort.InferenceSession(
        cfg.ONNX_SAVE_PATH,
        providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
    )
    dummy   = np.random.randn(1, 3, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE).astype(np.float32)
    outputs = session.run(None, {"face_crop": dummy})

    cls_out = outputs[0][0]
    vad_out = outputs[1][0]
    pred_class = CLASS_NAMES[int(np.argmax(cls_out))]

    print(f"Output 0 — cls_logits shape : {outputs[0].shape}")
    print(f"Output 1 — vad shape        : {outputs[1].shape}")
    print(f"Predicted class             : {pred_class}")
    print(f"VAD output                  : V={vad_out[0]:.4f}  A={vad_out[1]:.4f}  D={vad_out[2]:.4f}")
    print("ONNX verification passed.")

# main

In [11]:
model, history = train(cfg)
export_onnx(model, cfg)
verify_onnx(cfg)

history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(cfg.OUTPUT_DIR, "training_history.csv"), index=False)

print("\nFiles ready for download:")
print(f"  {cfg.ONNX_SAVE_PATH}")
print(f"  {cfg.MODEL_SAVE_PATH}")
print(f"  {cfg.OUTPUT_DIR}/training_history_xper01.csv")

Loaded 28,709 samples from: /kaggle/input/datasets/msambare/fer2013/train
Loaded 7,178 samples from: /kaggle/input/datasets/msambare/fer2013/test
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 141MB/s] 



Starting training...

 Epoch |  Phase |    Total |      CLS |      VAD |     Acc
----------------------------------------------------------
Backbone frozen — training heads only.



     1 |  train |   1.1406 |   1.7174 |   0.2754 |  31.61%
       |    val |   1.0760 |   1.6487 |   0.2169 |  36.95%
----------------------------------------------------------
  >> Best model saved (val acc: 36.95%)



     2 |  train |   1.0661 |   1.6209 |   0.2339 |  36.34%
       |    val |   1.0544 |   1.6104 |   0.2205 |  39.06%
----------------------------------------------------------
  >> Best model saved (val acc: 39.06%)



...

    36 |  train |   0.3449 |   0.5215 |   0.0798 |  80.76%
       |    val |   0.6507 |   1.0095 |   0.1126 |  68.39%
----------------------------------------------------------


    37 |  train |   0.3442 |   0.5211 |   0.0788 |  80.80%
       |    val |   0.6529 |   1.0130 |   0.1128 |  67.96%
----------------------------------------------------------


    38 |  train |   0.3400 |   0.5134 |   0.0798 |  81.10%
       |    val |   0.6507 |   1.0098 |   0.1121 |  68.40%
----------------------------------------------------------
  >> Best model saved (val acc: 68.40%)



    39 |  train |   0.3366 |   0.5086 |   0.0785 |  81.32%
       |    val |   0.6503 |   1.0094 |   0.1116 |  68.60%
----------------------------------------------------------
  >> Best model saved (val acc: 68.60%)



    40 |  train |   0.3387 |   0.5122 |   0.0785 |  81.03%
       |    val |   0.6511 |   1.0106 |   0.1119 |  68.31%
----------------------------------------------------------

Training complete. Best Val Accuracy: 68.60%
ONNX exported : /kaggle/working/vad_multitask.onnx (16.6 MB)
Output 0 — cls_logits shape : (1, 7)
Output 1 — vad shape        : (1, 3)
Predicted class             : sad
VAD output                  : V=-1.0000  A=-1.0000  D=-1.0000
ONNX verification passed.

Files ready for download:
  /kaggle/working/vad_multitask.onnx
  /kaggle/working/vad_multitask.pth
  /kaggle/working/training_history_xper01.csv
